# Cascade stage2a — buyin_direction

Binary classifier. Part of the 4-stage NS commitment cascade.

In [ ]:
!pip install -q -U transformers datasets scikit-learn

In [ ]:
import glob, json, random, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score, classification_report,
    confusion_matrix, precision_recall_fscore_support,
)
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, EarlyStoppingCallback, DataCollatorWithPadding,
)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


In [ ]:
def find_input_file(*names):
    for name in names:
        for pattern in [f"/kaggle/input/**/{name}", f"/kaggle/input/datasets/kevinnchan/**/{name}"]:
            matches = glob.glob(pattern, recursive=True)
            if matches:
                return Path(matches[0])
        if Path(name).exists():
            return Path(name)
    raise FileNotFoundError(f"Cannot find any of {names}")


In [ ]:
class CascadeDataset(Dataset):
    def __init__(self, encodings, label_ids, weights=None):
        self.encodings = encodings
        self.labels    = label_ids
        self.weights   = weights if weights is not None else [1.0] * len(label_ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"]  = torch.tensor(self.labels[idx], dtype=torch.long)
        item["weights"] = torch.tensor(self.weights[idx], dtype=torch.float)
        return item


In [ ]:
class WeightedCollator(DataCollatorWithPadding):
    def __call__(self, features):
        labels  = torch.tensor([f.pop("labels")           for f in features], dtype=torch.long)
        weights = torch.tensor([f.pop("weights", 1.0)     for f in features], dtype=torch.float)
        batch   = super().__call__(features)
        batch["labels"]  = labels
        batch["weights"] = weights
        return batch

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        weights = inputs.pop("weights")
        outputs = model(**inputs)
        loss_fn = nn.CrossEntropyLoss(reduction="none")
        loss    = (loss_fn(outputs.logits, labels) * weights).mean()
        return (loss, outputs) if return_outputs else loss


In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "kappa":    cohen_kappa_score(labels, preds),
        "accuracy": accuracy_score(labels, preds),
    }


In [ ]:
# ── Config ────────────────────────────────────────────────────────────────
STAGE      = "stage2a"
AXIS       = "buyin_direction"
MODEL_NAME = "zanelim/singbert-large-sg"   # replace with your saved SingBERT Kaggle dataset path if needed
MAX_LEN    = 128
BATCH_SIZE = 32
EPOCHS     = 6
LR         = 2e-05
GRAD_ACCUM = 2
VAL_FRAC   = 0.15

ID2LABEL   = {"0": "committed", "1": "uncommitted"}
LABEL2ID   = {"committed": 0, "uncommitted": 1}
NUM_LABELS = 2

OUTPUT_DIR = Path(f"/kaggle/working/{STAGE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Stage: {STAGE}  |  Axis: {AXIS}  |  Labels: {LABEL2ID}")


In [ ]:
# ── Load training data ────────────────────────────────────────────────────
train_path = find_input_file("stage2a_train.csv")
print(f"Loading: {train_path}")
df = pd.read_csv(train_path)
print(f"Total rows: {len(df):,}")
print(df["label"].value_counts().to_string())

# Encode labels
df = df[df["label"].notna()].copy()
# label column is already int (0/1) for binary stages
if df["label"].dtype == object:
    df["label_id"] = df["label"].map(LABEL2ID)
else:
    df["label_id"] = df["label"].astype(int)
df = df[df["label_id"].notna()].copy()
df["label_id"] = df["label_id"].astype(int)
df["weight"]   = df["weight"].fillna(1.0).astype(float)
df["text"]     = df["text"].fillna("").astype(str).str.strip()
df = df[df["text"].str.len() > 5].reset_index(drop=True)

train_df, val_df = train_test_split(
    df, test_size=VAL_FRAC, random_state=SEED, stratify=df["label_id"]
)
print(f"Train: {len(train_df):,}  Val: {len(val_df):,}")
print(f"Train label dist: {train_df['label_id'].value_counts().to_dict()}")


In [ ]:
# ── Tokenise ──────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_enc = tokenizer(train_df["text"].tolist(), truncation=True, max_length=MAX_LEN, padding=False)
val_enc   = tokenizer(val_df["text"].tolist(),   truncation=True, max_length=MAX_LEN, padding=False)

train_dataset = CascadeDataset(train_enc, train_df["label_id"].tolist(), train_df["weight"].tolist())
val_dataset   = CascadeDataset(val_enc,   val_df["label_id"].tolist(),   [1.0]*len(val_df))
print(f"Train dataset: {len(train_dataset)}  Val dataset: {len(val_dataset)}")


In [ ]:
# ── Model + Trainer ───────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS,
    id2label=ID2LABEL, label2id=LABEL2ID, ignore_mismatched_sizes=True,
)
model.to(DEVICE)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    save_total_limit            = 1,
    load_best_model_at_end      = True,
    metric_for_best_model       = "kappa",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    report_to                   = "none",
)

trainer = WeightedTrainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
    data_collator   = WeightedCollator(tokenizer),
)

train_result = trainer.train()
print(f"Best val kappa: {trainer.state.best_metric:.4f}")


In [ ]:
# ── Evaluate on commitment_testset.parquet ────────────────────────────────
test_path = find_input_file("commitment_testset.parquet")
test_df   = pd.read_parquet(test_path)

# Pick the right column for this axis
GOLD_COL = "human_label"
test_df = test_df[test_df[GOLD_COL].isin(LABEL2ID)].reset_index(drop=True)
print(f"Evaluable testset rows: {len(test_df)}  (labels: {LABEL2ID})")
print(test_df[GOLD_COL].value_counts().to_string())

test_enc = tokenizer(test_df["text"].tolist(), truncation=True, max_length=MAX_LEN, padding=False)
test_labels = test_df[GOLD_COL].map(LABEL2ID).tolist()
test_dataset = CascadeDataset(test_enc, test_labels)

out = trainer.predict(test_dataset)
probs = torch.softmax(torch.tensor(out.predictions, dtype=torch.float32), dim=-1).numpy()
preds = np.argmax(out.predictions, axis=-1)
preds_named = [ID2LABEL[str(int(p))] for p in preds]
true_named  = test_df[GOLD_COL].tolist()

print("\n" + "="*72)
print(f"  {STAGE} — {AXIS} — Testset Evaluation")
print("="*72)
print(f"  Accuracy: {accuracy_score(test_labels, preds):.1%}")
print(f"  Kappa:    {cohen_kappa_score(test_labels, preds):.3f}")
print()
print(classification_report(true_named, preds_named, digits=3))
print(confusion_matrix(true_named, preds_named))
print("="*72)

# Save predictions
eval_df = test_df.copy()
eval_df["pred_label"] = preds_named
for i, lbl in ID2LABEL.items():
    eval_df[f"prob_{lbl}"] = probs[:, int(i)]
eval_df.to_csv(OUTPUT_DIR / f"testset_eval_{STAGE}.csv", index=False)
print(f"Saved eval → {OUTPUT_DIR}/testset_eval_{STAGE}.csv")


In [ ]:
model_out = OUTPUT_DIR / "best_model"
trainer.save_model(str(model_out))
tokenizer.save_pretrained(str(model_out))
with open(model_out / "id2label.json", "w") as f:
    json.dump({"axis": AXIS, "id2label": ID2LABEL, "label2id": LABEL2ID}, f, indent=2)
print(f"Model saved → {model_out}")

zip_path = str(OUTPUT_DIR.parent / f"cascade_{AXIS.replace('/', '_')}")
shutil.make_archive(zip_path, "zip", str(model_out))
print(f"Zipped → {zip_path}.zip  ({Path(zip_path+'.zip').stat().st_size/1e6:.0f} MB)")
print("Upload this zip as a new Kaggle dataset for inference.")
